In [ ]:
# Imports
import os
from pathlib import Path
from urllib.parse import urlparse

from fastai.vision.all import *
from fastai.vision.gan import *
from fastai.vision.core import *

In [ ]:
# Check that we have our GPU
import torch
#cuda configs
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

print(f'CUDA is available: {torch.cuda.is_available()}')

In [ ]:
# Variables
input_path = "./data/silhouette/tatooless"

# Training batch size, tune this based on the amount of memory
batch_size = 4
# Lets get the largest image we can fit in memory with a reasonable batch size
image_size = (640,480) #512 #128
#image_size = (320,240)

# Util Functions

In [ ]:
# get_y function (from training the models)
def _get_y(x):
    return path_hr/x.name

In [ ]:
resize = Resize((training_height, training_width), 
                 resamples=(Image.BILINEAR, Image.NEAREST),
                 method=ResizeMethod.Pad,
                 pad_mode=PadMode.Zeros)

def predict_image(img):
    img_fast = resize(img, split_idx=1) # Resize the image to match our training.
    img_fast.show()
    
    # Run our prediction
    tensor_image, img_hr_tensor_base, preds_tensor_base = learn.predict(img_fast)
    
    # Show our results
    pil_image = PILImage.create(tensor_image)
    pil_image.show()
    
    return pil_image

# Model Loading

In [ ]:
# Load the model
model_file = './silhouette-xresnet34_deeper-epocs1000.pkl'
learn = load_learner(model_file)

# Process images

In [ ]:
raw_image_files = get_image_files(input_path)

# loop over all the images and process them
for raw_image_file in raw_image_files:
    # load the image and process
    print(f"Processing {raw_image_file}")
    basename = os.path.basename(raw_image_file)
    pil_image = PILImage.create(raw_image_file)
    mask_img = predict_image(pil_image)
    mask_img.save("./data/silhouette/mask-predict/" + re.sub(r'\.', "_mask.", basename))